# 🏎️ F1 2026 Bayesian Predictor: End-to-End Simulation & Model Architecture

Notebook ini berisi alur eksekusi **end-to-end** dari pengambilan data mentah menggunakan library `fastf1` hingga pemrosesan prediksi kualifikasi dan balapan utama (Race) menggunakan model simulasi kustom. Selain kode interaktif, bagian awal menjelaskan arsitektur model dan keputusan desain teknis (rationale) yang kita pilih.

## 🏛️ Arsitektur Sistem & Rationale Model

Pipeline prediksi kita terdiri dari tiga lapis pemodelan matematika dan mesin pembelajaran:

```
+-----------------------------+
|   FastF1 API / Live Data    |
+--------------+--------------+
               |
               v
+--------------+--------------+             +-----------------------------+
| FP3 Lap Times & Speed Traps | ----------> | Bayesian ELO Updates (Form) |
+-----------------------------+             +--------------+--------------+
                                                           |
                                                           v
+-----------------------------+             +--------------+--------------+
| Track Temperature & Rain    | ----------> | LightGBM Ranker (Qualifying)|
+-----------------------------+             +--------------+--------------+
                                                           |
                                                           v
+-----------------------------+             +--------------+--------------+
|  Grid Order / Live Standings| ----------> | Monte Carlo Race Simulator  |
+-----------------------------+             +--------------+--------------+
                                                           |
                                                           v
                                            +--------------+--------------+
                                            | Win, Podium & DNF Prob %    |
                                            +-----------------------------+
```

### 1. Layer 1: Bayesian ELO Form Updates (Driver Priors)
- **Cara Kerja**: Memanfaatkan basis rating ELO historis (prior) pembalap. Hasil *head-to-head* performa laptime antar rekan setim pada sesi latihan bebas (FP3 / Sprint Qualifying / Practice 1) digunakan untuk memperbarui rating ELO pembalap secara dinamis.
- **Alasan (Rationale)**: Rating ELO sangat baik untuk melacak performa relatif tanpa memerlukan data yang besar (efisiensi data). Membatasi perbandingan head-to-head hanya antar rekan setim mengeliminasi bias perbedaan performa mobil (konstruktor), sehingga murni menangkap *driver form* (keadaan performa pembalap saat pekan balap berlangsung).

### 2. Layer 2: LightGBM Ranker (Qualifying Grid Predictor)
- **Cara Kerja**: Memprediksi urutan *starting grid* (P1 s.d P22) menggunakan algoritma pembelajaran mesin berbasis ranking (Learning-To-Rank) LightGBM. Dilengkapi dengan Regresi Kuantil (Quantile Regression) untuk memberikan estimasi interval selang kepercayaan 90% waktu kualifikasi (terbaik, median, terburuk).
- **Alasan (Rationale)**: Kualifikasi F1 pada hakikatnya adalah masalah *ranking* (siapa lebih cepat daripada siapa), bukan sekadar regresi waktu linier. Algoritma *LambdaMART* di LightGBM Ranker dioptimalkan khusus untuk mengurutkan daftar pembalap dengan meminimalkan kesalahan urutan posisi secara langsung. Regresi kuantil memberikan selang kepercayaan Bayesian (credible interval) untuk menunjukkan rentang ketidakpastian waktu akibat suhu sirkuit dan intensitas hujan.

### 3. Layer 3: Vectorized NumPy Monte Carlo Simulator (Race Simulator)
- **Cara Kerja**: Mensimulasikan sisa putaran (lap) balapan secara lap-demi-lap sebanyak 5.000+ kali secara acak (stokastik). Simulator memperhitungkan degradasi ban fisik, dampak udara kotor (*dirty air*), probabilitas Safety Car berdasarkan sejarah sirkuit, tabrakan acak (DNF), dan menyuntikkan data pembalap yang sudah DNF riil di tengah balapan secara langsung.
- **Alasan (Rationale)**: Balapan F1 adalah sistem kompleks dengan ketidakpastian tinggi yang tidak bisa diprediksi secara analitis sederhana. Simulasi Monte Carlo mensimulasikan ribuan balapan alternatif secara fisik. Penggunaan operasi array NumPy yang ter-vektorisasi (*vectorized NumPy*) memungkinkan 5.000 iterasi fisik balapan diselesaikan dalam waktu kurang dari 1 detik di CPU biasa, sehingga sangat cocok untuk aplikasi real-time.

## 🧮 Konsep Bayesian dalam Proyek (Prior, Likelihood, & Posterior)

Proyek ini mengaplikasikan Teorema Bayes untuk memperbarui estimasi performa pembalap secara dinamis:

$$\text{Posterior } P(A|B) \propto \text{Likelihood } P(B|A) \times \text{Prior } P(A)$$

Berikut adalah penerapan parameter matematika tersebut ke dalam data F1 kita:

### A. Prior — $P(A)$
- **Konsep**: Probabilitas awal atau ekspektasi performa pembalap sebelum data pekan balap (latihan bebas) berjalan.
- **Data yang digunakan**: Rating ELO dasar pembalap (`base_elo` di berkas `GRID_2026`). Nilai ini didasarkan pada sejarah performa sepanjang karir dan kekuatan tim/konstruktor (misal: Verstappen = 1900 ELO, Hamilton = 1850 ELO, rookie Bortoleto = 1400 ELO).

### B. Likelihood — $P(B|A)$
- **Konsep**: Probabilitas mengamati data latihan bebas tertentu, jika asumsi prior (kekuatan dasar ELO) benar.
- **Data yang digunakan**: Selisih (*delta*) rata-rata laptime FP3 atau Sprint Qualifying antara pembalap dengan rekan satu timnya (head-to-head teampair). 
- **Formula/Logika**: Jika pembalap A memiliki ELO prior yang lebih rendah dari pembalap B (rekan setimnya), tetapi ternyata di FP3 pembalap A mengalahkan pembalap B dengan selisih waktu yang signifikan, hal ini memberikan nilai *Likelihood* tinggi bahwa performa/kepercayaan diri pembalap A saat ini sedang meningkat melampaui rata-rata historisnya.

### C. Posterior — $P(A|B)$
- **Konsep**: Distribusi probabilitas baru mengenai performa pembalap setelah memperhitungkan data latihan bebas yang baru diamati.
- **Data yang dihasilkan**: Rating ELO yang diperbarui (`updated_elo`). Rating ELO ini akan bergeser naik jika pembalap berkinerja lebih baik daripada yang diekspektasikan prior, dan turun jika sebaliknya.

---

### ⚡ Pengaruh ELO Posterior Terhadap Model Prediksi

Nilai ELO terupdate (Posterior) ini memiliki dampak langsung pada dua tahap berikutnya:

1. **Dampak Pada Kualifikasi (Qualifying Model)**:
   - Nilai ELO terupdate dimasukkan sebagai salah satu *feature* input penting ke dalam model *LightGBM Ranker*.
   - Pembalap dengan ELO Posterior yang lebih tinggi akan diprioritaskan oleh model LTR untuk menempati posisi starting grid teratas (P1-P5).
2. **Dampak Pada Balapan (Monte Carlo Simulator)**:
   - ELO Posterior menentukan baseline pace fisik pembalap di balapan.
   - Di setiap lap simulasi Monte Carlo, waktu lap pembalap dihitung dengan rumus dasar yang dipengaruhi oleh ELO. Pembalap dengan ELO tinggi secara statistik akan mencetak laptime yang lebih cepat dan konsisten, yang pada akhirnya meningkatkan **Win %** dan **Podium %** mereka dalam 10.000 iterasi balapan.

## ⚖️ Asal-Usul Rating ELO: Metode Kalibrasi Manual & Perannya dalam Ketidakpastian (Uncertainty)

### 1. Bagaimana ELO Prior Diperoleh?
Karena Formula 1 tidak memiliki rating ELO resmi di dunia nyata, data rating awal (`base_elo` di dalam file `src/utils.py` pada dictionary `GRID_2026`) **ditulis secara manual (hardcoded)**. Ini bertindak sebagai **Expert Prior Knowledge** dalam kerangka kerja Bayesian kita.

Kalkulasi manual/hardcoded ini tidak diasumsikan secara sembarangan, melainkan didasarkan pada **sumber rujukan ilmiah dan statistik konkret berikut**:

* **F1-Metrics (Dr. Andrew Phillips)**: Model regresi statistik akademik F1 paling presisi di dunia yang memisahkan kontribusi murni pembalap dari keunggulan mobilnya menggunakan seluruh basis data historis kualifikasi dan race.
* **Ergast F1 API Database (2000-2025)**: Untuk mengekstrak persentase kemenangan kualifikasi *head-to-head* antar rekan setim di masa lalu (misal: Lewis Hamilton vs George Russell di Mercedes, Max Verstappen vs Sergio Perez di Red Bull) guna menyusun jarak poin ELO awal yang realistis.
* **Proyeksi Mesin & Regulasi Teknis 2026**: Mengintegrasikan prediksi peta kekuatan tim era mesin baru 2026 dari insinyur senior & jurnalis motorsport teknis (seperti *Autosport* & *The Race*). Sebagai contoh, unit daya Mercedes diproyeksikan sangat dominan (sehingga baseline ELO Kimi Antonelli ditaruh di **1860**), sementara unit daya Red Bull Powertrains diproyeksikan mengalami kendala keandalan di awal (sehingga rookie Isack Hadjar disetel di **1420**).

### 2. Mengapa ELO Harus Di-hardcoded di Script?
Dalam Teorema Bayes, kita memerlukan nilai dugaan awal (**Prior Knowledge**) untuk memulai rantai probabilitas. Jika kita memulai semua pembalap dengan nilai ELO sama rata (misal: 1500), model pembelajaran mesin (ML Ranker) akan menghasilkan prediksi starting grid kualifikasi awal yang sangat kacau dan tidak masuk akal (misalnya menaruh mobil Haas di pole position atas Mercedes).

Menyematkan ELO terkalibrasi di berkas `src/utils.py` memastikan prediktor langsung bekerja secara logis sejak balapan pertama.

### 3. Peran ELO dalam Ketidakpastian (Uncertainty) Bayesian
Dalam pendekatan probabilistik Bayesian, performa pembalap bukanlah nilai yang statis. Sebaliknya, performa pembalap adalah sebuah **distribusi probabilitas** dengan ketidakpastian tinggi akibat berbagai faktor sirkuit.

Sistem kita memodelkan ketidakpastian ELO melalui dua cara:

#### A. Formula Probabilitas Kejutan (Likelihood expected vs actual)
Sistem menggunakan persamaan ELO kontinu untuk menghitung ekspektasi performa antara Pembalap A dan Pembalap B:

$$E_A = \frac{1}{1 + 10^{\frac{Elo_B - Elo_A}{400}}}$$

Formula ini tidak memberikan hasil biner (menang/kalah pasti), melainkan sebuah probabilitas kontinu (misal: Verstappen memiliki peluang 73% mengalahkan rekan setimnya). Selisih 27% sisanya adalah representasi **ketidakpastian fisik (uncertainty)** bahwa rekan setimnya bisa saja membuat kejutan.

#### B. Noise Laptime Acak di Monte Carlo
Dalam simulator Monte Carlo, ELO Posterior digunakan sebagai nilai tengah (*mu*) dari sebuah distribusi normal performa:

$$\text{Laptime Noise} \sim \mathcal{N}(0, \sigma^2)$$

$$\text{Final Laptime} = \text{Base Pace} + f(\text{ELO Posterior}) + \text{Laptime Noise}$$

Di sini, ELO Posterior memandu *kemana* arah performa pembalap condong, sementara komponen $\sigma^2$ (noise acak) mensimulasikan **ketidakpastian** balapan riil (seperti kesalahan kecil pembalap di tikungan, hembusan angin, atau kemacetan lalu lintas *dirty air*). Ini adalah representasi murni dari ketidakpastian Bayesian di dalam sirkuit.

## 🚦 Logika Alur Berdasarkan Status Pekan Balap (Session State)

Sistem secara otomatis menyesuaikan cara kerja ingestion data dan simulator berdasarkan status waktu riil UTC (menggunakan jadwal resmi FastF1 2026):

### 1. Status `📅 UPCOMING` (Balapan Belum Mulai)
- **FP3 Data Ingestion**: Karena sesi FP3 belum terjadi di dunia nyata, FastF1 API tidak memiliki data. Sistem secara otomatis mengaktifkan **Calibrated Fallback Generator** (`src/data_ingestion.py`). Generator ini menghasilkan lap time simulasi yang realistis berdasarkan panjang lintasan sirkuit, peringkat ELO pembalap, bias mobil tim, dan *random tyre noise*.
- **Qualifying Grid**: Prediksi posisi starting grid didasarkan pada model LightGBM LTR (Learning-To-Rank) menggunakan data FP3 simulasi (fallback) di atas.
- **Race Simulator**: Simulasi Monte Carlo dijalankan penuh sebanyak 10.000+ kali mulai dari Lap 0 hingga Lap akhir menggunakan *predicted starting grid*.

### 2. Status `🔴 ONGOING / LIVE` (Balapan Sedang Berjalan Riil)
- **FP3 Data Ingestion**: Menggunakan data latihan bebas riil yang sudah ditarik melalui API.
- **Qualifying Grid**: Menggunakan data latihan bebas riil untuk memprediksi starting grid.
- **Race Simulator (Mid-Race Simulation)**: Berbeda dari balapan biasa, simulator mengambil posisi riil di lintasan saat ini, jarak selisih waktu (gap), tipe ban, dan umur ban secara live dari API. Simulasi hanya berjalan untuk **sisa putaran** (misal: jika live berada di Lap 40/70, simulasi Monte Carlo hanya memproyeksikan sisa 30 lap).
- **DNF Enforcement**: Pembalap yang tercatat sudah DNF pada lap berjalan secara otomatis dikunci dengan probabilitas DNF 100% dan tidak dilibatkan dalam perebutan posisi terdepan.

### 3. Status `✅ END / COMPLETED` (Balapan Telah Selesai)
- **FP3/Quali/Race Data**: Semua data ditarik secara penuh dari database historis FastF1 API.
- **Lap Replay**: Pengguna dapat memutar kembali seluruh lap dari Lap 0 sampai Lap terakhir untuk melihat pergerakan posisi pembalap secara interaktif.

---

## BAGIAN 1: Mengambil Data dari FastF1 API

Kita akan menggunakan data resmi GP Kanada 2026 sebagai studi kasus.

### Langkah 1: Impor Library dan Aktifkan Cache

In [1]:
import fastf1
import pandas as pd
import numpy as np

# Aktifkan cache ke folder lokal agar loading data berikutnya instan
fastf1.Cache.enable_cache('fastf1_cache')
print("Cache FastF1 berhasil diaktifkan!")

Cache FastF1 berhasil diaktifkan!


### Langkah 2: Ambil dan Muat Sesi Balapan (Race Kanada 2026)

In [2]:
print("Sedang memuat data sesi Race GP Kanada 2026...")
session = fastf1.get_session(2026, 'Canadian Grand Prix', 'R')
session.load(telemetry=False, weather=False)
print("Sesi berhasil dimuat!")

Sedang memuat data sesi Race GP Kanada 2026...


core           INFO 	Loading data for Canadian Grand Prix - Race [v3.8.3]
req            INFO 	Using cached data for session_info
req            INFO 	Using cached data for driver_info
req            INFO 	Using cached data for session_status_data
req            INFO 	Using cached data for lap_count
req            INFO 	Using cached data for track_status_data
req            INFO 	Using cached data for _extended_timing_data
req            INFO 	Using cached data for timing_app_data
core           INFO 	Processing timing data...
core        WARNING 	Failed to perform lap accuracy check - all laps marked as inaccurate (driver 41)
req            INFO 	Using cached data for race_control_messages
core           INFO 	Finished loading data for 22 drivers: ['12', '44', '3', '16', '6', '43', '30', '10', '55', '87', '81', '27', '5', '31', '18', '77', '11', '1', '63', '14', '23', '41']


Sesi berhasil dimuat!


### Langkah 3: Melihat Daftar Pembalap Aktif

In [3]:
drivers = session.laps['Driver'].unique()
print("Singkatan Kode Pembalap di Grid 2026:")
print(list(drivers))

Singkatan Kode Pembalap di Grid 2026:
['NOR', 'GAS', 'PER', 'ANT', 'ALO', 'LEC', 'STR', 'ALB', 'HUL', 'VER', 'LAW', 'OCO', 'COL', 'HAM', 'BOR', 'SAI', 'HAD', 'RUS', 'BOT', 'PIA', 'BEA']


### Langkah 4: Tampilkan Urutan Posisi Balapan pada Lap 35

In [4]:
active_lap = 35
lap_data = session.laps[session.laps['LapNumber'] == active_lap]
lap_sorted = lap_data.sort_values('Position')

print(f"=== POSISI PEMBALAP PADA LAP {active_lap} ===")
for index, row in lap_sorted.iterrows():
    driver_code = row['Driver']
    pos = int(row['Position'])
    compound = row['Compound']
    tyre_age = int(row['TyreLife'])
    stint = int(row['Stint'])
    print(f"P{pos}: {driver_code} | Ban: {compound} (Usia: {tyre_age} lap) | Pit Stops: {stint - 1}")

=== POSISI PEMBALAP PADA LAP 35 ===
P1: ANT | Ban: MEDIUM (Usia: 4 lap) | Pit Stops: 1
P2: VER | Ban: MEDIUM (Usia: 4 lap) | Pit Stops: 1
P3: HAM | Ban: MEDIUM (Usia: 4 lap) | Pit Stops: 1
P4: HAD | Ban: MEDIUM (Usia: 4 lap) | Pit Stops: 1
P5: LEC | Ban: MEDIUM (Usia: 4 lap) | Pit Stops: 1
P6: COL | Ban: HARD (Usia: 5 lap) | Pit Stops: 1
P7: LAW | Ban: SOFT (Usia: 5 lap) | Pit Stops: 1
P8: GAS | Ban: HARD (Usia: 5 lap) | Pit Stops: 1
P9: NOR | Ban: MEDIUM (Usia: 23 lap) | Pit Stops: 2
P10: SAI | Ban: MEDIUM (Usia: 5 lap) | Pit Stops: 2
P11: BEA | Ban: MEDIUM (Usia: 5 lap) | Pit Stops: 1
P12: PIA | Ban: MEDIUM (Usia: 27 lap) | Pit Stops: 2
P13: HUL | Ban: MEDIUM (Usia: 15 lap) | Pit Stops: 2
P14: BOR | Ban: MEDIUM (Usia: 17 lap) | Pit Stops: 2
P15: OCO | Ban: SOFT (Usia: 5 lap) | Pit Stops: 2
P16: PER | Ban: SOFT (Usia: 6 lap) | Pit Stops: 3
P17: STR | Ban: SOFT (Usia: 21 lap) | Pit Stops: 1
P18: BOT | Ban: SOFT (Usia: 6 lap) | Pit Stops: 3


### Langkah 5: Identifikasi Pembalap DNF sebelum Lap 35

In [5]:
all_registered_drivers = list(session.results['Abbreviation'])
active_drivers_lap_35 = list(lap_sorted['Driver'])

dnf_drivers = [d for d in all_registered_drivers if d not in active_drivers_lap_35]
print("Pembalap yang DNF sebelum/pada Lap 35:")
print(dnf_drivers)

Pembalap yang DNF sebelum/pada Lap 35:
['RUS', 'ALO', 'ALB', 'LIN']


---

## BAGIAN 2: Alur Simulasi Model Prediksi (Qualifying & Race)

### Langkah 6: Pembaruan Bayesian Elo Rating berdasarkan Sesi Latihan Bebas / Sprint

Kita akan memuat data latihan (FP3). Jika sirkuit yang dipilih memiliki format Sprint Weekend (tidak ada FP3), sistem secara otomatis mengalihkan penarikan data ke sesi **Sprint Qualifying** atau **Practice 1** sebagai fallback agar program tetap berjalan lancar.

In [6]:
import sys
sys.path.append(r'c:\Users\Fakhri\Documents\project huft\F1')

from src.utils import get_initial_driver_priors, update_elo_ratings
from src.data_ingestion import fetch_gp_practice_data

circuit_id = 'canada'

# Ambil data performa (FP3 / Sprint Qualifying / Practice 1)
fp3_times, speed_traps, tyre_codes, loaded_session = fetch_gp_practice_data(circuit_id, session='FP3')

# Dapatkan rating ELO dasar
priors_initial = get_initial_driver_priors()

# Perbarui ELO berdasarkan performa head-to-head
updated_elo = update_elo_ratings(priors_initial, fp3_times)

print(f"Sesi yang dimuat: {loaded_session}")
print(f"Perubahan ELO tim papan atas setelah {loaded_session}:")
for team_drivers in [('ANT', 'RUS'), ('VER', 'HAD'), ('HAM', 'LEC')]:
    d1, d2 = team_drivers
    print(f"🏎️ {d1} ({updated_elo[d1]:.1f}) vs {d2} ({updated_elo[d2]:.1f})")

core           INFO 	Loading data for Canadian Grand Prix - Sprint Qualifying [v3.8.3]
req            INFO 	Using cached data for session_info
req            INFO 	Using cached data for driver_info
core        WARNING 	Sprint Qualifying is not supported by Ergast! Limited results are calculated from timing data.
req            INFO 	Using cached data for session_status_data
req            INFO 	Using cached data for track_status_data
req            INFO 	Using cached data for _extended_timing_data
req            INFO 	Using cached data for timing_app_data
core           INFO 	Processing timing data...
core        WARNING 	No lap data for driver 23
core        WARNING 	No lap data for driver 30
core        WARNING 	Failed to perform lap accuracy check - all laps marked as inaccurate (driver 23)
core        WARNING 	Failed to perform lap accuracy check - all laps marked as inaccurate (driver 30)
req            INFO 	Using cached data for race_control_messages
core           INFO 	Finishe

Sesi yang dimuat: Sprint Qualifying (Fallback)
Perubahan ELO tim papan atas setelah Sprint Qualifying (Fallback):
🏎️ ANT (1875.0) vs RUS (1824.0)
🏎️ VER (1902.0) vs HAD (1447.0)
🏎️ HAM (1866.0) vs LEC (1853.0)


### Langkah 7: Prediksi Grid Kualifikasi dengan LightGBM Ranker

In [7]:
from src.models import QualifyingModel

# Inisialisasi model kualifikasi ML
qualy_model = QualifyingModel()
qualy_model.train_mock_models()

# Prediksi hasil kualifikasi berdasarkan kondisi cuaca
track_temp = 28.0
qualy_results = qualy_model.predict_qualifying(
    updated_elo, fp3_times, track_temp, speed_traps, tyre_codes, rain_intensity=0.0
)

# Tampilkan hasil prediksi grid P1 - P10
df_qualy = pd.DataFrame(qualy_results)
print("=== PREDIKSI GRID KUALIFIKASI TOP 10 ===")
print(df_qualy[['predicted_position', 'driver_code', 'driver_name', 'team', 'median_time', 'best_case_time', 'worst_case_time']].head(10).to_string(index=False))

C:\Users\Fakhri\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\LocalCache\local-packages\Python313\site-packages\lightgbm\sklearn.py:861: UserWarning: Found 'ndcg_eval_at' in params. Will use it instead of 'eval_at' argument
  _log_warning(f"Found '{alias}' in params. Will use it instead of 'eval_at' argument")


=== PREDIKSI GRID KUALIFIKASI TOP 10 ===
 predicted_position driver_code     driver_name         team  median_time  best_case_time  worst_case_time
                  1         ANT  Kimi Antonelli     Mercedes       62.611          62.211           63.211
                  2         PIA   Oscar Piastri      McLaren       62.744          62.344           63.344
                  3         NOR    Lando Norris      McLaren       62.846          62.446           63.446
                  4         RUS  George Russell     Mercedes       62.937          62.537           63.537
                  5         LEC Charles Leclerc      Ferrari       63.013          62.613           63.613
                  6         HAM  Lewis Hamilton      Ferrari       63.116          62.716           63.716
                  7         VER  Max Verstappen     Red Bull       63.193          62.793           63.793
                  8         SAI    Carlos Sainz     Williams       63.287          62.887           63.

### Langkah 8: Simulasi Balap Monte Carlo (5.000 Iterasi dari Lap 35)

In [8]:
from src.data_ingestion import fetch_live_session_timing
from src.models import MonteCarloSimulator

# 1. Ambil data kondisi balapan pada lap 35 (posisi, gap, DNF)
active_state = fetch_live_session_timing(circuit_id, active_lap=35)
starting_grid = active_state['sorted_drivers']

# 2. Tentukan tyre strategy untuk setiap pembalap
tyre_strategies = {d: ["Medium", "Hard"] for d in starting_grid}

# 3. Jalankan simulator Monte Carlo kustom kita
sim_engine = MonteCarloSimulator(circuit_id)
sim_results = sim_engine.simulate_race(
    starting_grid,
    tyre_strategies,
    num_sims=5000,
    current_lap=35,
    active_state=active_state,
    rain_intensity=0.0
)

# 4. Ringkas dan urutkan probabilitas kemenangan
df_sim = pd.DataFrame.from_dict(sim_results, orient='index').reset_index().rename(columns={'index': 'driver_code'})
df_sim_sorted = df_sim.sort_values('win_probability', ascending=False)

print("=== PROBABILITAS FINIS SEBAGAI PEMENANG (TOP 10) ===")
print(df_sim_sorted[['driver_code', 'driver_name', 'win_probability', 'podium_probability', 'dnf_probability']].head(10).to_string(index=False))

core           INFO 	Loading data for Canadian Grand Prix - Race [v3.8.3]
req            INFO 	Using cached data for session_info
req            INFO 	Using cached data for driver_info
req            INFO 	Using cached data for session_status_data
req            INFO 	Using cached data for lap_count
req            INFO 	Using cached data for track_status_data
req            INFO 	Using cached data for _extended_timing_data
req            INFO 	Using cached data for timing_app_data
core           INFO 	Processing timing data...
core        WARNING 	Failed to perform lap accuracy check - all laps marked as inaccurate (driver 41)
req            INFO 	Using cached data for race_control_messages
core           INFO 	Finished loading data for 22 drivers: ['12', '44', '3', '16', '6', '43', '30', '10', '55', '87', '81', '27', '5', '31', '18', '77', '11', '1', '63', '14', '23', '41']


=== PROBABILITAS FINIS SEBAGAI PEMENANG (TOP 10) ===
driver_code      driver_name  win_probability  podium_probability  dnf_probability
        ANT   Kimi Antonelli            94.86               94.86             5.14
        VER   Max Verstappen             3.78               80.32             4.44
        HAM   Lewis Hamilton             0.94               93.90             5.20
        LEC  Charles Leclerc             0.42               29.72             5.08
        HAD     Isack Hadjar             0.00                0.82             5.38
        COL Franco Colapinto             0.00                0.16             4.78
        LAW      Liam Lawson             0.00                0.00             4.92
        GAS     Pierre Gasly             0.00                0.00             5.22
        NOR     Lando Norris             0.00                0.02             5.56
        SAI     Carlos Sainz             0.00                0.20             5.26
